In [18]:
## Step 1 - Imports ##

import pandas as pd
import numpy as np

import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

In [19]:
## Step 2 - Load the data ##

# Load well datasets
well1 = pd.read_csv("1_2021002_.csv")
well4 = pd.read_csv("4_20211022_.csv")
well7 = pd.read_csv("7_2021022_.csv")

In [20]:
## Step 3 - Clean and combine ##

# Fix column issues
well4 = well4.rename(columns={"litholody": "lithology"})
well7 = well7.rename(columns={"litholody": "lithology"})

# Add well labels
well1["well"] = "well1"
well4["well"] = "well4"
well7["well"] = "well7"

# Combine
all_data = pd.concat([well1, well4, well7], axis=0)

# Keep relevant columns (no depth again)
columns = ["SP", "GR", "LLD", "LLS", "DEN", "AC", "lithology", "well"]
all_data = all_data[columns]

# Remove missing lithology
all_data = all_data.dropna(subset=["lithology"])

In [21]:
## Step 4 - Features ##

features = ["SP", "GR", "LLD", "LLS", "DEN", "AC"]

In [22]:
## Step 5 - Split by well ##

train_data = all_data[all_data["well"] != "well7"]
test_data = all_data[all_data["well"] == "well7"]

In [23]:
## Step 6 - Build X and y ##

X_train = train_data[features]
X_test = test_data[features]

# Fill missing values
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_train.mean())

# Encode lithology
le = LabelEncoder()

y_train = le.fit_transform(train_data["lithology"])
y_test = le.transform(test_data["lithology"])

In [24]:
## Step 7 - Add constant ##

X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

In [25]:
## Step 8 - Fit Logistic Model - statsmodels  ##

logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.351021
         Iterations 8


In [29]:
## Step 9 - Print Full Summary ##

print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                20050
Model:                          Logit   Df Residuals:                    20043
Method:                           MLE   Df Model:                            6
Date:                Fri, 26 Jun 2026   Pseudo R-squ.:                  0.3502
Time:                        02:14:23   Log-Likelihood:                -7038.0
converged:                       True   LL-Null:                       -10831.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -14.4911      0.919    -15.764      0.000     -16.293     -12.689
SP            -0.0419      0.001    -31.381      0.000      -0.045      -0.039
GR             0.0189      0.003      6.863      0.0

In [30]:
## Step 10 - Predictions ##

y_pred_prob = result.predict(X_test_sm)

# Convert probabilities → 0 or 1
y_pred = (y_pred_prob >= 0.5).astype(int)

In [31]:
## Step 11 - Accuracy ##

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.878089716203845


In [32]:
## Step 12 - Confusion Matrix ##

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1576  700]
 [  99 4179]]


In [33]:
## Interpretation

# Logistic regression was used to model the probability of lithology based on well log measurements. The coefficients indicate the effect of each feature on lithology classification. Positive coefficients increase the likelihood of shale, while negative coefficients increase the likelihood of sand. The model achieved moderate accuracy, indicating that linear models are less effective than tree-based models for capturing complex geological relationships.

# The logistic regression model was used to predict lithology based on well log measurements. The model achieved strong performance, with a pseudo R² of approximately 0.35 and statistically significant results (p < 0.001). Among the features, density (DEN) exhibited the strongest positive influence on the probability of shale, followed by gamma ray (GR). In contrast, spontaneous potential (SP), shallow resistivity (LLS), and acoustic log (AC) showed negative relationships, increasing the likelihood of sand. The LLD variable was found to be statistically insignificant, indicating it does not contribute meaningfully to the model. Overall, the results demonstrate that logistic regression can effectively capture the relationship between log measurements and lithology, although it is less flexible than tree-based models in capturing nonlinear behavior.


## Key Takeaways

# Model is statistically significant ✅
# DEN is the strongest predictor ✅
# GR confirms shale identification ✅
# SP, LLS, AC help identify sand ✅
# LLD is not useful ❌
# Logistic Regression performs well (~88%) ✅

# Logistic regression achieved approximately 88% accuracy in predicting lithology, demonstrating that a linear model can effectively capture relationships between well log measurements and rock types, although slightly less accurately than tree-based methods.

# Model is linear -> simpler interpretation